# End-to-end solar interferometer simulation (610 MHz)

This notebook replaces `notebooks/legacy/the_newest_heart_of_project.ipynb`. All physics lives in the
`interfsim` package (`src/interfsim`); the notebook only configures, runs and plots.

Install once from the repository root: `pip install -e ".[notebooks]"`.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import interfsim as ix
from interfsim.pipeline import PipelineConfig, run_pipeline

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = PipelineConfig.from_toml(ROOT / "configs" / "thesis_case1_20250320.toml")
cfg.observation.n_samples = 20_000      # lighter for interactive use; the config uses 50 000
cfg.integration_bins = 20               # keep the dump time (~38 s) and hence the fringe smearing unchanged
cfg.output.dir = "results/notebook_case1"
cfg.output.fpga_hex = False

## 1. Case 1 - four-antenna E-W array, Learmonth flux of 2025-03-20

In [ ]:
res = run_pipeline(cfg)
{k: res.summary[k] for k in ("hour_angle_range_deg", "snr_peak_db_per_sample", "dump_time_s",
                              "max_fringe_rotation_per_dump_rad", "psf_fwhm_ew_arcmin", "psf_fwhm_ns_arcmin",
                              "engineA_vs_engineB_image_rms_diff", "excess_sim_avg_db", "excess_measured_db")}

In [ ]:
from interfsim import plotting
plotting.plot_track(res); plotting.plot_uv(res); plotting.plot_image_psf(res); plt.show()

## 2. Layout comparison (thesis Cases 1, 1.5, 1.75, 2) with the forward model

The forward model (Engine B) is cheap, so it is the right tool for sweeping geometries. The PSF FWHM is
measured on the main lobe only (`metrics.fwhm_main_lobe`).

In [ ]:
from interfsim.metrics import psf_fwhm, uv_filling_factor
obs = ix.Observation(res.obs.lat_deg, res.obs.dec_deg, 610e6, np.linspace(res.obs.H_deg[0], res.obs.H_deg[-1], 300))
layouts = [ix.preset_compact(), ix.preset_cross4(), ix.preset_dlite(), ix.preset_extended()]
sky = {"type": "disk", "diameter_rad": np.radians(0.533)}
out = ix.sweep_layouts(layouts, obs, sky, fov_deg=3.0, grid=128)
fig, ax = plt.subplots(2, len(layouts), figsize=(4 * len(layouts), 8))
for k, a in enumerate(layouts):
    r = out[a.name]; L = r["l_deg"] * 60
    ew, ns = psf_fwhm(r["psf"], L, L)
    uvmax = np.hypot(r["u"], r["v"]).max()
    ax[0, k].imshow(r["image"], extent=[L[0], L[-1], L[0], L[-1]], origin="lower", cmap="hot")
    ax[0, k].set_title(f"{a.name} | FWHM E-W {ew:.0f}', N-S {ns:.0f}'", fontsize=9)
    ax[1, k].plot(np.r_[r["u"].ravel(), -r["u"].ravel()], np.r_[r["v"].ravel(), -r["v"].ravel()], "o", ms=0.8)
    ax[1, k].set_aspect("equal", adjustable="datalim"); ax[1, k].set_xlabel("u (wavelengths)")
    ax[1, k].set_title(f"{a.unique_baseline_directions()} baseline directions, fill {uv_filling_factor(r['u'], r['v'], uvmax):.2f}", fontsize=9)
plt.tight_layout(); plt.show()

## 3. Validation - extended Sun in the voltage engine reproduces van Cittert-Zernike

With `components=disk_components(...)` every surface element of the solar disk radiates an independent
noise process, so long baselines decorrelate. The measured |V| must follow the analytic jinc.

In [ ]:
from scipy.special import j1
diam = np.radians(0.533)
arr = ix.linear(12, 6.0, az_deg=90)
obs0 = ix.Observation(-22.22, 0.0, 610e6, np.zeros(40_000))   # frozen sky: stationary statistics
V = ix.simulate_voltages(arr, obs0, 1.0, 0.0, seed=1, components=ix.disk_components(diam, 21))
i, j = np.array(arr.pairs).T
vis = (V[:, i] * np.conj(V[:, j])).mean(0)
q = arr.baseline_lengths / obs0.lam
qq = np.linspace(1, q.max(), 400); x = np.pi * diam * qq
plt.plot(q, np.abs(vis), "o", ms=3, label="Engine A (voltages, disk components)")
plt.plot(qq, np.abs(2 * j1(x) / x), label="analytic |jinc|, 0.533 deg disk")
plt.xlabel("baseline (wavelengths)"); plt.ylabel("|V| / V(0)"); plt.legend(); plt.show()

## 4. Hardware comparison and FPGA stimulus

`res.summary` already contains the gain-independent solar excess for the Hoa Lac measurement
(`[comparison]` in the config). Set `cfg.output.fpga_hex = True` to write `iq_array_stimulus.txt`
(16-bit two's-complement I/Q per antenna) plus a metadata file.

In [ ]:
s = res.summary
print(f"simulated excess (|H|<=30 deg, mean): {s['excess_sim_avg_db']:.3f} dB")
print(f"measured excess  (Hoa Lac)          : {s['excess_measured_db']:.3f} dB")
print(f"ratio (linear)                      : {s['excess_ratio_linear']:.1f}x  ->  T_sys,eff ~ {150 * s['excess_ratio_linear']:.0f} K")